In [ ]:
import pandas as pd
import numpy as np
import seaborn as sn
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.linear_model import LinearRegression, LogisticRegression 
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, r2_score,
roc_auc_score, roc_curve, mean_squared_error, mean_absolute_error, root_mean_squared_error)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier 
from sklearn.decomposition import PCA

In [ ]:
!pip install xgboost
from xgboost import XGBClassifier

In [ ]:
!pip install lightgbm
from lightgbm import LGBMClassifier

In [ ]:
df_train=pd.read_csv(r"D:\DATA\DA syllabus\Home Credit Default Risk\application_train.csv")

In [ ]:
df_test=pd.read_csv(r"D:\DATA\DA syllabus\Home Credit Default Risk\application_test.csv")

In [ ]:
df_prev=pd.read_csv(r"D:\DATA\DA syllabus\Home Credit Default Risk\previous_application.csv")

In [ ]:
df_credit=pd.read_csv(r"D:\DATA\DA syllabus\Home Credit Default Risk\credit_card_balance.csv")

In [ ]:
df_bur=pd.read_csv(r"D:\DATA\DA syllabus\Home Credit Default Risk\bureau.csv")

In [ ]:
df_bur_bal=pd.read_csv(r"D:\DATA\DA syllabus\Home Credit Default Risk\bureau_balance.csv")

In [ ]:
df_inst=pd.read_csv(r"D:\DATA\DA syllabus\Home Credit Default Risk\installments_payments.csv")

In [ ]:
df_cash=pd.read_csv(r"D:\DATA\DA syllabus\Home Credit Default Risk\POS_CASH_balance.csv")

In [ ]:
list(df_train.columns)

In [ ]:
doc_cols = [col for col in df_train.columns if 'FLAG_DOCUMENT' in col]
df_train['TOTAL_DOCS'] = df_train[doc_cols].sum(axis=1)
df_train.drop(columns=doc_cols, inplace=True)

doc_cols = [col for col in df_test.columns if 'FLAG_DOCUMENT' in col]
df_test['TOTAL_DOCS'] = df_test[doc_cols].sum(axis=1)
df_test.drop(columns=doc_cols, inplace=True)

In [ ]:
df_train = df_train.drop(["FLAG_MOBIL", "FLAG_EMP_PHONE", "FLAG_WORK_PHONE", "FLAG_CONT_MOBILE", "FLAG_PHONE", "FLAG_EMAIL",
                          "REGION_RATING_CLIENT_W_CITY","APARTMENTS_AVG", "BASEMENTAREA_AVG", "YEARS_BEGINEXPLUATATION_AVG", "YEARS_BUILD_AVG", 
                          "COMMONAREA_AVG", "ELEVATORS_AVG", "ENTRANCES_AVG", "FLOORSMAX_AVG", "FLOORSMIN_AVG", "LANDAREA_AVG", 
                          "LIVINGAPARTMENTS_AVG", "LIVINGAREA_AVG", "NONLIVINGAPARTMENTS_AVG", "NONLIVINGAREA_AVG","APARTMENTS_MODE", 
                          "BASEMENTAREA_MODE", "YEARS_BEGINEXPLUATATION_MODE", "YEARS_BUILD_MODE", "COMMONAREA_MODE", "ELEVATORS_MODE", 
                          "ENTRANCES_MODE", "FLOORSMAX_MODE", "FLOORSMIN_MODE", "LANDAREA_MODE", "LIVINGAPARTMENTS_MODE", "LIVINGAREA_MODE", 
                          "NONLIVINGAPARTMENTS_MODE", "NONLIVINGAREA_MODE","APARTMENTS_MEDI", "BASEMENTAREA_MEDI", "YEARS_BEGINEXPLUATATION_MEDI", 
                          "YEARS_BUILD_MEDI", "COMMONAREA_MEDI", "ELEVATORS_MEDI", "ENTRANCES_MEDI", "FLOORSMAX_MEDI", "FLOORSMIN_MEDI", 
                          "LANDAREA_MEDI", "LIVINGAPARTMENTS_MEDI", "LIVINGAREA_MEDI", "NONLIVINGAPARTMENTS_MEDI", "NONLIVINGAREA_MEDI",
                          "FONDKAPREMONT_MODE", "HOUSETYPE_MODE", "TOTALAREA_MODE", "WALLSMATERIAL_MODE", "EMERGENCYSTATE_MODE"],axis=1)

df_test = df_test.drop(["FLAG_MOBIL", "FLAG_EMP_PHONE", "FLAG_WORK_PHONE", "FLAG_CONT_MOBILE", "FLAG_PHONE", "FLAG_EMAIL",
                          "REGION_RATING_CLIENT_W_CITY","APARTMENTS_AVG", "BASEMENTAREA_AVG", "YEARS_BEGINEXPLUATATION_AVG", "YEARS_BUILD_AVG", 
                          "COMMONAREA_AVG", "ELEVATORS_AVG", "ENTRANCES_AVG", "FLOORSMAX_AVG", "FLOORSMIN_AVG", "LANDAREA_AVG", 
                          "LIVINGAPARTMENTS_AVG", "LIVINGAREA_AVG", "NONLIVINGAPARTMENTS_AVG", "NONLIVINGAREA_AVG","APARTMENTS_MODE", 
                          "BASEMENTAREA_MODE", "YEARS_BEGINEXPLUATATION_MODE", "YEARS_BUILD_MODE", "COMMONAREA_MODE", "ELEVATORS_MODE", 
                          "ENTRANCES_MODE", "FLOORSMAX_MODE", "FLOORSMIN_MODE", "LANDAREA_MODE", "LIVINGAPARTMENTS_MODE", "LIVINGAREA_MODE", 
                          "NONLIVINGAPARTMENTS_MODE", "NONLIVINGAREA_MODE","APARTMENTS_MEDI", "BASEMENTAREA_MEDI", "YEARS_BEGINEXPLUATATION_MEDI", 
                          "YEARS_BUILD_MEDI", "COMMONAREA_MEDI", "ELEVATORS_MEDI", "ENTRANCES_MEDI", "FLOORSMAX_MEDI", "FLOORSMIN_MEDI", 
                          "LANDAREA_MEDI", "LIVINGAPARTMENTS_MEDI", "LIVINGAREA_MEDI", "NONLIVINGAPARTMENTS_MEDI", "NONLIVINGAREA_MEDI",
                          "FONDKAPREMONT_MODE", "HOUSETYPE_MODE", "TOTALAREA_MODE", "WALLSMATERIAL_MODE", "EMERGENCYSTATE_MODE"],axis=1)

In [ ]:
df_test.columns

In [ ]:
# ===== FEATURE ENGINEERING =====

# Financial ratios
df_train['INCOME_CREDIT_RATIO'] = df_train['AMT_INCOME_TOTAL'] / df_train['AMT_CREDIT']
df_train['ANNUITY_INCOME_RATIO'] = df_train['AMT_ANNUITY'] / df_train['AMT_INCOME_TOTAL']
df_train['CREDIT_GOODS_RATIO'] = df_train['AMT_CREDIT'] / df_train['AMT_GOODS_PRICE']

# Family burden
df_train['CHILDREN_RATIO'] = df_train['CNT_CHILDREN'] / df_train['CNT_FAM_MEMBERS']

# Age & employment
df_train['AGE_YEARS'] = -df_train['DAYS_BIRTH'] / 365
df_train['EMPLOYMENT_YEARS'] = -df_train['DAYS_EMPLOYED'] / 365

# Stability
df_train['PHONE_CHANGE_YEARS'] = -df_train['DAYS_LAST_PHONE_CHANGE'] / 365

# EXT_SOURCE combination
df_train['EXT_SOURCE_SUM'] = df_train[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']].sum(axis=1)

# Bureau requests summary
bureau_cols = [col for col in df_train.columns if 'AMT_REQ_CREDIT_BUREAU' in col]
df_train['TOTAL_BUREAU_REQUESTS'] = df_train[bureau_cols].sum(axis=1)

In [ ]:
# ===== FEATURE ENGINEERING =====

# Financial ratios
df_test['INCOME_CREDIT_RATIO'] = df_test['AMT_INCOME_TOTAL'] / df_test['AMT_CREDIT']
df_test['ANNUITY_INCOME_RATIO'] = df_test['AMT_ANNUITY'] / df_test['AMT_INCOME_TOTAL']
df_test['CREDIT_GOODS_RATIO'] = df_test['AMT_CREDIT'] / df_test['AMT_GOODS_PRICE']

# Family burden
df_test['CHILDREN_RATIO'] = df_test['CNT_CHILDREN'] / df_test['CNT_FAM_MEMBERS']

# Age & employment
df_test['AGE_YEARS'] = -df_test['DAYS_BIRTH'] / 365
df_test['EMPLOYMENT_YEARS'] = -df_test['DAYS_EMPLOYED'] / 365

# Stability
df_test['PHONE_CHANGE_YEARS'] = -df_test['DAYS_LAST_PHONE_CHANGE'] / 365

# EXT_SOURCE combination
df_test['EXT_SOURCE_SUM'] = df_test[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']].sum(axis=1)

# Bureau requests summary
bureau_cols = [col for col in df_test.columns if 'AMT_REQ_CREDIT_BUREAU' in col]
df_test['TOTAL_BUREAU_REQUESTS'] = df_test[bureau_cols].sum(axis=1)

In [ ]:
df_train.columns

In [ ]:
df_train.isnull().sum()

In [ ]:
df_train['AMT_ANNUITY'] = df_train['AMT_ANNUITY'].fillna(df_train['AMT_ANNUITY'].median())
df_train['AMT_GOODS_PRICE'] = df_train['AMT_GOODS_PRICE'].fillna(df_train['AMT_CREDIT'])
df_train['NAME_TYPE_SUITE'] = df_train['NAME_TYPE_SUITE'].fillna('Unknown')
df_train['OWN_CAR_AGE'] = df_train['OWN_CAR_AGE'].fillna(-1)
df_train['OCCUPATION_TYPE'] = df_train['OCCUPATION_TYPE'].fillna('Unknown')
df_train['CNT_FAM_MEMBERS'] = df_train['CNT_FAM_MEMBERS'].fillna(df_train['CNT_FAM_MEMBERS'].median())
df_train['DAYS_LAST_PHONE_CHANGE'] = df_train['DAYS_LAST_PHONE_CHANGE'].fillna(df_train['DAYS_LAST_PHONE_CHANGE'].median())

In [ ]:
df_train['EXT_SOURCE_MEAN'] = df_train[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']].mean(axis=1)

for col in ['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']:
    df_train[col] = df_train[col].fillna(df_train['EXT_SOURCE_MEAN'])
    df_train[col] = df_train[col].fillna(0)

In [ ]:
bureau_cols = [col for col in df_train.columns if 'AMT_REQ_CREDIT_BUREAU' in col]
df_train[bureau_cols] = df_train[bureau_cols].fillna(0)

In [ ]:
social_cols = [
    'OBS_30_CNT_SOCIAL_CIRCLE',
    'DEF_30_CNT_SOCIAL_CIRCLE',
    'OBS_60_CNT_SOCIAL_CIRCLE',
    'DEF_60_CNT_SOCIAL_CIRCLE'
]

df_train[social_cols] = df_train[social_cols].fillna(0)

In [ ]:
df_train['EXT_SOURCE_MEAN'] = df_train['EXT_SOURCE_MEAN'].fillna(0)

In [ ]:
df_train.isnull().sum()

In [ ]:
df_test['AMT_ANNUITY'] = df_test['AMT_ANNUITY'].fillna(df_test['AMT_ANNUITY'].median())
df_test['AMT_GOODS_PRICE'] = df_test['AMT_GOODS_PRICE'].fillna(df_test['AMT_CREDIT'])
df_test['NAME_TYPE_SUITE'] = df_test['NAME_TYPE_SUITE'].fillna('Unknown')
df_test['OWN_CAR_AGE'] = df_test['OWN_CAR_AGE'].fillna(-1)
df_test['OCCUPATION_TYPE'] = df_test['OCCUPATION_TYPE'].fillna('Unknown')
df_test['CNT_FAM_MEMBERS'] = df_test['CNT_FAM_MEMBERS'].fillna(df_test['CNT_FAM_MEMBERS'].median())
df_test['DAYS_LAST_PHONE_CHANGE'] = df_test['DAYS_LAST_PHONE_CHANGE'].fillna(df_test['DAYS_LAST_PHONE_CHANGE'].median())

In [ ]:
df_test['EXT_SOURCE_MEAN'] = df_test[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']].mean(axis=1)

for col in ['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']:
    df_test[col] = df_test[col].fillna(df_test['EXT_SOURCE_MEAN'])
    df_test[col] = df_test[col].fillna(0)

In [ ]:
bureau_cols1 = [col for col in df_test.columns if 'AMT_REQ_CREDIT_BUREAU' in col]
df_test[bureau_cols] = df_test[bureau_cols1].fillna(0)

In [ ]:
social_cols1 = [
    'OBS_30_CNT_SOCIAL_CIRCLE',
    'DEF_30_CNT_SOCIAL_CIRCLE',
    'OBS_60_CNT_SOCIAL_CIRCLE',
    'DEF_60_CNT_SOCIAL_CIRCLE'
]

df_test[social_cols1] = df_test[social_cols1].fillna(0)

In [ ]:
df_test['EXT_SOURCE_MEAN'] = df_test['EXT_SOURCE_MEAN'].fillna(0)

In [ ]:
df_test.isnull().sum()

In [ ]:
df_train['SK_ID_CURR'].duplicated().sum()

In [ ]:
df_test['SK_ID_CURR'].duplicated().sum()

In [ ]:
#df_prev, df_credit, df_bur, df_bur_bal, df_inst, df_cash
df_cash.duplicated().sum()

In [ ]:
#df_prev, df_credit, df_bur, df_bur_bal, df_inst, df_cash
df_cash.isnull().sum()

In [ ]:
df_cash['CNT_INSTALMENT'] = df_cash['CNT_INSTALMENT'].fillna(df_cash['CNT_INSTALMENT'].median())
df_cash['CNT_INSTALMENT_FUTURE'] = df_cash['CNT_INSTALMENT_FUTURE'].fillna(df_cash['CNT_INSTALMENT_FUTURE'].median())

In [ ]:
df_cash['LATE_PAYMENT'] = (df_cash['SK_DPD'] > 0).astype(int)
df_cash['SERIOUS_LATE'] = (df_cash['SK_DPD_DEF'] > 0).astype(int)

In [ ]:
pos_agg = df_cash.groupby('SK_ID_CURR').agg({
    'SK_ID_PREV': 'nunique',
    'MONTHS_BALANCE': ['max', 'min'],
    'CNT_INSTALMENT': ['mean', 'sum'],
    'CNT_INSTALMENT_FUTURE': 'mean',
    'SK_DPD': ['max', 'mean'],
    'SK_DPD_DEF': ['max'],
    'LATE_PAYMENT': 'mean',
    'SERIOUS_LATE': 'mean'
})

In [ ]:
pos_agg.columns = ['_'.join(col).upper() for col in pos_agg.columns]
pos_agg.reset_index(inplace=True)

In [ ]:
#df_prev, df_credit, df_bur, df_bur_bal, df_inst, df_cash
df_prev.isnull().sum()

In [ ]:
cols_to_drop = [
    'RATE_INTEREST_PRIMARY',
    'RATE_INTEREST_PRIVILEGED',
    'AMT_DOWN_PAYMENT',
    'RATE_DOWN_PAYMENT',
    'DAYS_FIRST_DRAWING',
    'DAYS_FIRST_DUE',
    'DAYS_LAST_DUE_1ST_VERSION',
    'DAYS_LAST_DUE',
    'DAYS_TERMINATION',
    'NFLAG_INSURED_ON_APPROVAL'
]

df_prev.drop(columns=cols_to_drop, inplace=True)

In [ ]:
df_prev['AMT_ANNUITY'] = df_prev['AMT_ANNUITY'].fillna(df_prev['AMT_ANNUITY'].median())
df_prev['AMT_GOODS_PRICE'] = df_prev['AMT_GOODS_PRICE'].fillna(df_prev['AMT_CREDIT'])
df_prev['CNT_PAYMENT'] = df_prev['CNT_PAYMENT'].fillna(df_prev['CNT_PAYMENT'].median())

# Categorical
df_prev['NAME_TYPE_SUITE'] = df_prev['NAME_TYPE_SUITE'].fillna('Unknown')
df_prev['PRODUCT_COMBINATION'] = df_prev['PRODUCT_COMBINATION'].fillna('Unknown')

In [ ]:
df_prev['APP_CREDIT_RATIO'] = df_prev['AMT_APPLICATION'] / df_prev['AMT_CREDIT']
df_prev['ANNUITY_CREDIT_RATIO'] = df_prev['AMT_ANNUITY'] / df_prev['AMT_CREDIT']

# Approved / refused flags
df_prev['APPROVED'] = (df_prev['NAME_CONTRACT_STATUS'] == 'Approved').astype(int)
df_prev['REFUSED'] = (df_prev['NAME_CONTRACT_STATUS'] == 'Refused').astype(int)

In [ ]:
prev_agg = df_prev.groupby('SK_ID_CURR').agg({
    'SK_ID_PREV': 'nunique',
    'AMT_APPLICATION': 'mean',
    'AMT_CREDIT': ['mean', 'max'],
    'AMT_ANNUITY': 'mean',
    'CNT_PAYMENT': ['mean', 'sum'],
    'DAYS_DECISION': 'max',
    'APPROVED': 'mean',
    'REFUSED': 'mean',
    'APP_CREDIT_RATIO': 'mean',
    'ANNUITY_CREDIT_RATIO': 'mean'
})

In [ ]:
prev_agg.columns = ['_'.join(col).upper() for col in prev_agg.columns]
prev_agg.reset_index(inplace=True)

In [ ]:
#df_prev, df_credit, df_bur, df_bur_bal, df_inst, df_cash
df_credit.isnull().sum()

In [ ]:
draw_cols = [
    'AMT_DRAWINGS_ATM_CURRENT',
    'AMT_DRAWINGS_OTHER_CURRENT',
    'AMT_DRAWINGS_POS_CURRENT',
    'CNT_DRAWINGS_ATM_CURRENT',
    'CNT_DRAWINGS_OTHER_CURRENT',
    'CNT_DRAWINGS_POS_CURRENT'
]

df_credit[draw_cols] = df_credit[draw_cols].fillna(0)

In [ ]:
df_credit['AMT_PAYMENT_CURRENT'] = df_credit['AMT_PAYMENT_CURRENT'].fillna(0)
df_credit['AMT_INST_MIN_REGULARITY'] = df_credit['AMT_INST_MIN_REGULARITY'].fillna(0)
df_credit['CNT_INSTALMENT_MATURE_CUM'] = df_credit['CNT_INSTALMENT_MATURE_CUM'].fillna(0)

In [ ]:
df_credit['UTILIZATION'] = df_credit['AMT_BALANCE'] / (df_credit['AMT_CREDIT_LIMIT_ACTUAL'] + 1)
df_credit['PAYMENT_RATIO'] = df_credit['AMT_PAYMENT_CURRENT'] / (df_credit['AMT_TOTAL_RECEIVABLE'] + 1)
df_credit['DRAWING_RATIO'] = df_credit['AMT_DRAWINGS_CURRENT'] / (df_credit['AMT_CREDIT_LIMIT_ACTUAL'] + 1)
df_credit['LATE_PAYMENT'] = (df_credit['SK_DPD'] > 0).astype(int)
df_credit['SERIOUS_LATE'] = (df_credit['SK_DPD_DEF'] > 0).astype(int)

In [ ]:
df_credit.isnull().sum()

In [ ]:
credit_agg = df_credit.groupby('SK_ID_CURR').agg({
    'SK_ID_PREV': 'nunique',
    'MONTHS_BALANCE': ['max', 'min'],
    'AMT_BALANCE': ['mean', 'max', 'sum'],
    'AMT_CREDIT_LIMIT_ACTUAL': ['mean','sum'],
    'AMT_DRAWINGS_CURRENT': 'sum',
    'AMT_PAYMENT_CURRENT': 'sum',
    'AMT_TOTAL_RECEIVABLE': 'mean',
    'UTILIZATION': ['mean', 'max'],
    'PAYMENT_RATIO': 'mean',
    'DRAWING_RATIO': 'mean',
    'SK_DPD': ['mean', 'max'],
    'SK_DPD_DEF': 'max',
    'LATE_PAYMENT': 'mean',
    'SERIOUS_LATE': 'mean'
})

In [ ]:
credit_agg.columns = ['_'.join(col).upper() for col in credit_agg.columns]
credit_agg.reset_index(inplace=True)

In [ ]:
#df_prev, df_credit, df_bur, df_bur_bal, df_inst, df_cash
df_bur.isnull().sum()

In [ ]:
cols_to_drop = [
    'AMT_CREDIT_MAX_OVERDUE',
    'AMT_ANNUITY'
]

df_bur.drop(columns=cols_to_drop, inplace=True)

In [ ]:
df_bur['DAYS_CREDIT_ENDDATE'] = df_bur['DAYS_CREDIT_ENDDATE'].fillna(0)
df_bur['DAYS_ENDDATE_FACT'] = df_bur['DAYS_ENDDATE_FACT'].fillna(0)
df_bur['AMT_CREDIT_SUM'] = df_bur['AMT_CREDIT_SUM'].fillna(0)
df_bur['AMT_CREDIT_SUM_DEBT'] = df_bur['AMT_CREDIT_SUM_DEBT'].fillna(0)
df_bur['AMT_CREDIT_SUM_LIMIT'] = df_bur['AMT_CREDIT_SUM_LIMIT'].fillna(0)

In [ ]:
df_bur['DEBT_RATIO'] = df_bur['AMT_CREDIT_SUM_DEBT'] / df_bur['AMT_CREDIT_SUM']
df_bur['HAS_OVERDUE'] = (df_bur['AMT_CREDIT_SUM_OVERDUE'] > 0).astype(int)
df_bur['IS_ACTIVE'] = (df_bur['CREDIT_ACTIVE'] == 'Active').astype(int)
df_bur.replace([np.inf, -np.inf], np.nan, inplace=True)
df_bur['DEBT_RATIO'] = df_bur['DEBT_RATIO'].fillna(0)

In [ ]:
bureau_agg = df_bur.groupby('SK_ID_CURR').agg({
    'SK_ID_BUREAU': 'nunique',
    'DAYS_CREDIT': ['mean', 'min', 'max'],
    'CREDIT_DAY_OVERDUE': ['mean', 'max'],
    'AMT_CREDIT_SUM': ['mean', 'sum'],
    'AMT_CREDIT_SUM_DEBT': ['mean', 'sum'],
    'AMT_CREDIT_SUM_OVERDUE': ['sum'],
    'CNT_CREDIT_PROLONG': 'sum',
    'DEBT_RATIO': 'mean',
    'HAS_OVERDUE': 'mean',
    'IS_ACTIVE': 'mean'
})

In [ ]:
bureau_agg.columns = ['_'.join(col).upper() for col in bureau_agg.columns]
bureau_agg.reset_index(inplace=True)

In [ ]:
#df_prev, df_credit, df_bur, df_bur_bal, df_inst, df_cash
df_bur_bal.info()

In [ ]:
df_bur_bal['STATUS'].value_counts()

In [ ]:
# Late flag (1–5 = late)
df_bur_bal['LATE'] = df_bur_bal['STATUS'].isin(['1','2','3','4','5']).astype(int)

# Severe late (3+)
df_bur_bal['SEVERE_LATE'] = df_bur_bal['STATUS'].isin(['3','4','5']).astype(int)

# Closed flag
df_bur_bal['CLOSED'] = (df_bur_bal['STATUS'] == 'C').astype(int)

In [ ]:
bb_agg = df_bur_bal.groupby('SK_ID_BUREAU').agg({
    'MONTHS_BALANCE': ['min', 'max'],
    'LATE': 'mean',
    'SEVERE_LATE': 'mean',
    'CLOSED': 'mean'
})

In [ ]:
bb_agg.columns = ['_'.join(col).upper() for col in bb_agg.columns]
bb_agg.reset_index(inplace=True)

df_bur = df_bur.merge(bb_agg, on='SK_ID_BUREAU', how='left')

In [ ]:
bb_final = df_bur.groupby('SK_ID_CURR').agg({
    'LATE_MEAN': 'mean',
    'SEVERE_LATE_MEAN': 'mean',
    'CLOSED_MEAN': 'mean'
})

In [ ]:
bb_final.columns = ['BB_' + col.upper() for col in bb_final.columns]
bb_final.reset_index(inplace=True)

In [ ]:
df_inst.duplicated().sum()

In [ ]:
df_inst['DAYS_ENTRY_PAYMENT'] = df_inst['DAYS_ENTRY_PAYMENT'].fillna(df_inst['DAYS_INSTALMENT'])
df_inst['AMT_PAYMENT'] = df_inst['AMT_PAYMENT'].fillna(0)

In [ ]:
df_inst['PAYMENT_DELAY'] = df_inst['DAYS_ENTRY_PAYMENT'] - df_inst['DAYS_INSTALMENT']
#df_inst['LATE_PAYMENT'] = (df_inst['PAYMENT_DELAY'] > 0).astype(int)
#df_inst['PAYMENT_DIFF'] = df_inst['AMT_PAYMENT'] - df_inst['AMT_INSTALMENT']
df_inst['PAYMENT_RATIO'] = df_inst['AMT_PAYMENT'] / df_inst['AMT_INSTALMENT']
df_inst.replace([np.inf, -np.inf], np.nan, inplace=True)
df_inst['PAYMENT_RATIO'] = df_inst['PAYMENT_RATIO'].fillna(0)

In [ ]:
ins_agg = df_inst.groupby('SK_ID_CURR').agg({
    'NUM_INSTALMENT_VERSION': 'nunique',
    'NUM_INSTALMENT_NUMBER': 'max',
    'PAYMENT_DELAY': ['mean', 'max'],
    'PAYMENT_RATIO': 'mean',
    'AMT_INSTALMENT': ['sum','mean'],
    'AMT_PAYMENT': 'sum'
})

In [ ]:
ins_agg.columns = ['_'.join(col).upper() for col in ins_agg.columns]
ins_agg.reset_index(inplace=True)

In [ ]:
df_train['is_train'] = 1
df_test['is_train'] = 0

full = pd.concat([df_train, df_test], axis=0)

In [ ]:
full = full.merge(pos_agg, on='SK_ID_CURR', how='left')
full = full.merge(prev_agg, on='SK_ID_CURR', how='left')
full = full.merge(credit_agg, on='SK_ID_CURR', how='left')
full = full.merge(bureau_agg, on='SK_ID_CURR', how='left')
full = full.merge(bb_final, on='SK_ID_CURR', how='left')
full = full.merge(ins_agg, on='SK_ID_CURR', how='left')

In [ ]:
train_final = full[full['is_train'] == 1]
test_final = full[full['is_train'] == 0]

In [ ]:
train_final.drop(columns=['is_train'], inplace=True)
test_final.drop(columns=['is_train'], inplace=True)

In [ ]:
train_final.to_csv('train_final.csv', index=False)
test_final.to_csv('test_final.csv', index=False)

In [ ]:
train_final.isnull().sum()

In [ ]:
pd.set_option('display.max_rows', None)
train_final.isnull().sum()

In [ ]:
for col in train_final.columns:
    if col.endswith('_x'):
        base = col[:-2]
        if base + '_y' in train_final.columns:
            print(base)

In [ ]:
train_final[['PAYMENT_RATIO_MEAN_x', 'PAYMENT_RATIO_MEAN_y']].describe()

In [ ]:
train_final[['PAYMENT_RATIO_MEAN_x', 'PAYMENT_RATIO_MEAN_y']].isnull().sum()

In [ ]:
(train_final[['PAYMENT_RATIO_MEAN_x', 'PAYMENT_RATIO_MEAN_y']] == 0).sum()

In [ ]:
train_final[['PAYMENT_RATIO_MEAN_x', 'PAYMENT_RATIO_MEAN_y', 'TARGET']].corr()

In [ ]:
train_final.drop(columns=['PAYMENT_RATIO_MEAN_y'], inplace=True)
test_final.drop(columns=['PAYMENT_RATIO_MEAN_y'], inplace=True)

train_final.rename(columns={'PAYMENT_RATIO_MEAN_x': 'PAYMENT_RATIO_MEAN'}, inplace=True)
test_final.rename(columns={'PAYMENT_RATIO_MEAN_x': 'PAYMENT_RATIO_MEAN'}, inplace=True)

SK_ID_PREV_NUNIQUE  
MONTHS_BALANCE_MAX  
MONTHS_BALANCE_MIN  
SK_DPD_MAX  
SK_DPD_MEAN  
SK_DPD_DEF_MAX  
LATE_PAYMENT_MEAN  
SERIOUS_LATE_MEAN  
PAYMENT_RATIO_MEAN  

train_final[['PAYMENT_RATIO_MEAN_x', 'PAYMENT_RATIO_MEAN_y']].describe()  
train_final[['PAYMENT_RATIO_MEAN_x', 'PAYMENT_RATIO_MEAN_y']].isnull().sum()  
(train_final[['PAYMENT_RATIO_MEAN_x', 'PAYMENT_RATIO_MEAN_y']] == 0).sum()  
train_final[['PAYMENT_RATIO_MEAN_x', 'PAYMENT_RATIO_MEAN_y', 'TARGET']].corr()  
train_final.drop(columns=['PAYMENT_RATIO_MEAN_y'], inplace=True)  
test_final.drop(columns=['PAYMENT_RATIO_MEAN_y'], inplace=True)  

train_final.rename(columns={'PAYMENT_RATIO_MEAN_x': 'PAYMENT_RATIO_MEAN'}, inplace=True)  
test_final.rename(columns={'PAYMENT_RATIO_MEAN_x': 'PAYMENT_RATIO_MEAN'}, inplace=True)  

In [ ]:
train_final[['SK_ID_PREV_NUNIQUE_x', 'SK_ID_PREV_NUNIQUE_y']].describe()

In [ ]:
train_final[['SK_ID_PREV_NUNIQUE_x', 'SK_ID_PREV_NUNIQUE_y']].isnull().sum()

In [ ]:
train_final.drop(columns=['SK_ID_PREV_NUNIQUE_x'], inplace=True)
test_final.drop(columns=['SK_ID_PREV_NUNIQUE_x'], inplace=True)

train_final.rename(columns={'SK_ID_PREV_NUNIQUE_y': 'SK_ID_PREV_NUNIQUE'}, inplace=True)
test_final.rename(columns={'SK_ID_PREV_NUNIQUE_y': 'SK_ID_PREV_NUNIQUE'}, inplace=True)

In [ ]:
train_final[['MONTHS_BALANCE_MAX_x', 'MONTHS_BALANCE_MAX_y']].describe()

In [ ]:
train_final[['MONTHS_BALANCE_MAX_x', 'MONTHS_BALANCE_MAX_y']].isnull().sum()

In [ ]:
train_final[['MONTHS_BALANCE_MAX_x', 'MONTHS_BALANCE_MAX_y', 'TARGET']].corr()

In [ ]:
train_final[['MONTHS_BALANCE_MIN_x', 'MONTHS_BALANCE_MIN_y', 'TARGET']].corr()

In [ ]:
train_final.drop(columns=['MONTHS_BALANCE_MIN_y'], inplace=True)
test_final.drop(columns=['MONTHS_BALANCE_MIN_y'], inplace=True)

train_final.rename(columns={'MONTHS_BALANCE_MIN_x': 'MONTHS_BALANCE_MIN'}, inplace=True)
test_final.rename(columns={'MONTHS_BALANCE_MIN_x': 'MONTHS_BALANCE_MIN'}, inplace=True)

train_final.drop(columns=['MONTHS_BALANCE_MAX_y'], inplace=True)
test_final.drop(columns=['MONTHS_BALANCE_MAX_y'], inplace=True)

train_final.rename(columns={'MONTHS_BALANCE_MAX_x': 'MONTHS_BALANCE_MAX'}, inplace=True)
test_final.rename(columns={'MONTHS_BALANCE_MAX_x': 'MONTHS_BALANCE_MAX'}, inplace=True)

In [ ]:
train_final[['SK_DPD_MEAN_x', 'SK_DPD_MEAN_y']].describe()

In [ ]:
train_final[['SK_DPD_MEAN_x', 'SK_DPD_MEAN_y']].isnull().sum()

In [ ]:
(train_final[['SK_DPD_MEAN_x', 'SK_DPD_MEAN_y']] == 0).sum()

In [ ]:
train_final[['SK_DPD_MEAN_x', 'SK_DPD_MEAN_y', 'TARGET']].corr()

In [ ]:
train_final[['SK_DPD_MAX_x', 'SK_DPD_MAX_y', 'TARGET']].corr()

In [ ]:
train_final.drop(columns=['SK_DPD_MAX_y'], inplace=True)
test_final.drop(columns=['SK_DPD_MAX_y'], inplace=True)

train_final.rename(columns={'SK_DPD_MAX_x': 'SK_DPD_MAX'}, inplace=True)
test_final.rename(columns={'SK_DPD_MAX_x': 'SK_DPD_MAX'}, inplace=True)

train_final.drop(columns=['SK_DPD_MEAN_y'], inplace=True)
test_final.drop(columns=['SK_DPD_MEAN_y'], inplace=True)

train_final.rename(columns={'SK_DPD_MEAN_x': 'SK_DPD_MEAN'}, inplace=True)
test_final.rename(columns={'SK_DPD_MEAN_x': 'SK_DPD_MEAN'}, inplace=True)

In [ ]:
train_final[['SK_DPD_DEF_MAX_x', 'SK_DPD_DEF_MAX_y']].describe()

In [ ]:
train_final[['SK_DPD_DEF_MAX_x', 'SK_DPD_DEF_MAX_y']].isnull().sum()

In [ ]:
(train_final[['SK_DPD_DEF_MAX_x', 'SK_DPD_DEF_MAX_y']] == 0).sum()

In [ ]:
train_final[['SK_DPD_DEF_MAX_x', 'SK_DPD_DEF_MAX_y', 'TARGET']].corr()

In [ ]:
train_final.drop(columns=['SK_DPD_DEF_MAX_y'], inplace=True)
test_final.drop(columns=['SK_DPD_DEF_MAX_y'], inplace=True)

train_final.rename(columns={'SK_DPD_DEF_MAX_x': 'SK_DPD_DEF_MAX'}, inplace=True)
test_final.rename(columns={'SK_DPD_DEF_MAX_x': 'SK_DPD_DEF_MAX'}, inplace=True)

In [ ]:
train_final[['LATE_PAYMENT_MEAN_x', 'LATE_PAYMENT_MEAN_y']].describe()

In [ ]:
train_final[['LATE_PAYMENT_MEAN_x', 'LATE_PAYMENT_MEAN_y']].isnull().sum()

In [ ]:
(train_final[['LATE_PAYMENT_MEAN_x', 'LATE_PAYMENT_MEAN_y']] == 0).sum()

In [ ]:
train_final[['LATE_PAYMENT_MEAN_x', 'LATE_PAYMENT_MEAN_y', 'TARGET']].corr()

In [ ]:
train_final[['SERIOUS_LATE_MEAN_x', 'SERIOUS_LATE_MEAN_y', 'TARGET']].corr()

In [ ]:
train_final.drop(columns=[
    'LATE_PAYMENT_MEAN_y',
    'SERIOUS_LATE_MEAN_y'
], inplace=True)

test_final.drop(columns=[
    'LATE_PAYMENT_MEAN_y',
    'SERIOUS_LATE_MEAN_y'
], inplace=True)

train_final.rename(columns={
    'LATE_PAYMENT_MEAN_x': 'LATE_PAYMENT_MEAN',
    'SERIOUS_LATE_MEAN_x': 'SERIOUS_LATE_MEAN'
}, inplace=True)

test_final.rename(columns={
    'LATE_PAYMENT_MEAN_x': 'LATE_PAYMENT_MEAN',
    'SERIOUS_LATE_MEAN_x': 'SERIOUS_LATE_MEAN'
}, inplace=True)

In [ ]:
train_final.isnull().sum()

In [ ]:
train_final.info()

In [ ]:
test_final.isnull().sum()

In [ ]:
test_final.info()

In [ ]:
train_final.fillna(0, inplace=True)
test_final.fillna(0, inplace=True)

In [ ]:
train_final.to_csv('train_final.csv', index=False)
test_final.to_csv('test_final.csv', index=False)

In [ ]:
train_final.isnull().sum()

In [ ]:
train_final.columns

In [ ]:
df_train['AMT_BALANCE_SUM'].describe()